# fos_counts (cleaned) — spatial semi-NMF factorization of the whole-brain morphine c-Fos data

Reproducible driver that produces the **spatial clustering results** consumed by manuscript
**Figure 6** (`Fig6_semiNMF.py`) and by Figures 8 / S16 (`factor{i}.npy`).

Pipeline: load the per-subject c-Fos count heatmaps -> downsample -> fit a Poisson semi-NMF
model (`fos.seminmf_counts_2`) -> save `params.pkl`, `params.mat`, `npy/factor{k}.npy`, and
`factors.zarr`.

This is the *factorization step only*. The downstream factor statistics / spatial-panel
analysis lives in the manuscript repo (`Fig6_semiNMF.py`) and is not duplicated here.

Paths resolve from environment variables so the notebook is shareable:
`OPIOID_DATA_ROOT` (the Figshare deposit) and, optionally, `OPIOID_FACTOR_RESULTS`
(where to write; defaults to a timestamped folder under the group).


In [ ]:
import os
import pickle
from datetime import datetime

import numpy as np
import pandas as pd
import tifffile
import dask.array as da
import zarr

import jax.numpy as jnp
from skimage.transform import resize
from scipy.ndimage import zoom
from scipy.io import savemat

from fos import seminmf_counts_2 as seminmf

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ---- Figshare deposit layout ----
DATA_ROOT = os.environ.get("OPIOID_DATA_ROOT", "/path/to/Figshare_deposit")
GROUP = os.path.join(DATA_ROOT, "01_main_cfos_morphine")
ATLAS = os.path.join(DATA_ROOT, "shared", "atlas")

# Where to write the factorization outputs. Figure 6 reads these from OPIOID_FACTOR_RESULTS
# (else GROUP/spatial_clustering_results).
RESULTS_DIR = os.environ.get(
    "OPIOID_FACTOR_RESULTS",
    os.path.join(GROUP, "spatial_clustering_results", datetime.now().strftime("%Y_%m_%d-%H_%M")),
)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("writing results to", RESULTS_DIR)

In [ ]:
# annotated atlas + contour map (Kim reference)
atlas_img = tifffile.imread(os.path.join(ATLAS, "Kim_ref_adult_FP-label_v4.0.tif"))
contour_img = tifffile.imread(os.path.join(ATLAS, "Kim_ref_adult_FP-label_v2.9_contour_map.tif"))

In [ ]:
# meta (morphine conditions only)
Conditions = ['Saline', 'Acute_Morphine', 'Chronic_Morphine',
              'Withdrawal_Morphine', 'Chronic_Morphine_21', 'Withdrawal_Morphine_21']
metadf = pd.read_csv(os.path.join(GROUP, "OP_meta.csv"))
metadf = metadf[metadf.Condition.isin(Conditions)]

In [ ]:
# per-subject c-Fos count heatmaps (rows aligned to fnamelist)
counts = da.from_zarr(os.path.join(GROUP, "OP_cFos_heatmap_array"), mode="r")
fnamelist = np.load(os.path.join(GROUP, "OP_cFos_fnamelist.npy"))
assert len(fnamelist) == counts.shape[0]
metadf = metadf.set_index('fname').loc[fnamelist].reset_index()

In [ ]:
counts_3d = counts.reshape([counts.shape[0]] + list(atlas_img.shape)).compute()
assert counts_3d.shape[0] == 43
drugs = metadf.Condition.values

In [ ]:
# Raw heatmaps are 20x20x50 um; downsample to 50x50x50 um for the factorization.
# `input_voxel_shape` keeps the ORIGINAL (full-res) grid so factors can be upsampled back on save.
input_voxel_size = (20, 20, 50)
target_voxel_size = (50, 50, 50)
input_voxel_shape = atlas_img.shape
target_voxel_shape = tuple(np.array(
    np.array(input_voxel_shape) * np.array(input_voxel_size)[::-1] / np.array(target_voxel_size)[::-1],
    dtype='int'))

atlas_img = resize(atlas_img, output_shape=target_voxel_shape, order=0,
                   mode='reflect', anti_aliasing=True).astype(np.int16)
counts_3d = np.array([resize(f, output_shape=target_voxel_shape, order=0,
                             mode='reflect', anti_aliasing=False) for f in counts_3d]).astype(np.int16)

In [ ]:
# keep only voxels with signal in at least one subject
alive_voxels = np.sum(counts_3d, axis=0) != 0
print(np.sum(alive_voxels), "/", np.prod(counts_3d.shape[1:]), "voxels are 'alive'")
counts = counts_3d[:, alive_voxels]

## Fit the Poisson semi-NMF model

The number of factors (**22**) and sparsity penalty (**1e-2**) were chosen by cross-validated
model selection (sparsity in {1e-4, 1e-3, 1e-2, 1e-1, 1e0}, factors in {8, 10, ..., 24};
held-out log-likelihood on random spatial masks). That sweep was tracked with Weights & Biases;
here we fit directly with the selected values to regenerate the deposited result. The full sweep
code remains in the original notebook / git history.


In [ ]:
mean_func = "softplus"
best_num_factors = 22
best_sparsity_penalty = 1e-2
elastic_net_frac = 1.0
num_iters = 500
num_coord_ascent_iters = 1

initial_params = seminmf.initialize_nnsvd(counts, best_num_factors, mean_func, drugs=None)
params, losses, heldout_loglikes = seminmf.fit_poisson_seminmf(
    counts, initial_params, mask=None, mean_func=mean_func,
    sparsity_penalty=best_sparsity_penalty, elastic_net_frac=elastic_net_frac,
    num_iters=num_iters, num_coord_ascent_iters=num_coord_ascent_iters, tolerance=1e-5,
)
print("final loss:", float(losses[-1]))

In [ ]:
# --- save the fitted parameters ---
with open(os.path.join(RESULTS_DIR, "params.pkl"), "wb") as f:
    pickle.dump(params, f)

savemat(os.path.join(RESULTS_DIR, "params.mat"),
        dict(factors=np.array(params.factors),
             count_loadings=np.array(params.count_loadings),
             count_row_effects=np.array(params.count_row_effects),
             count_col_effects=np.array(params.count_col_effects)))

In [ ]:
# --- save each factor as a full-resolution volume, then pack into a zarr array ---
def upsample(arr, target_shape):
    zoom_factors = [t / c for t, c in zip(target_shape, arr.shape)]
    return zoom(arr, zoom_factors, order=0)  # nearest-neighbor

def save_factor(factor, filename):
    arr = jnp.zeros(alive_voxels.shape)
    arr = arr.at[alive_voxels].set(factor)
    arr = arr / arr.max()
    arr = upsample(np.array(arr), input_voxel_shape)  # back to full-res atlas grid
    np.save(filename, arr)

npy_dir = os.path.join(RESULTS_DIR, "npy")
os.makedirs(npy_dir, exist_ok=True)
for k, factor in enumerate(params.factors):
    save_factor(factor, os.path.join(npy_dir, f"factor{k}.npy"))

In [ ]:
first_factor = np.load(os.path.join(npy_dir, "factor0.npy"))
factor_shape = first_factor.flatten().shape
z = zarr.open(os.path.join(RESULTS_DIR, "factors.zarr"), mode="w",
              shape=(best_num_factors, *factor_shape),
              chunks=(1, *factor_shape), dtype=np.float32)
for k in range(best_num_factors):
    z[k] = np.load(os.path.join(npy_dir, f"factor{k}.npy")).flatten()
print("wrote", os.path.join(RESULTS_DIR, "factors.zarr"))